In [4]:
#!/usr/bin/env python3
import os
import torch, numpy as np
from PIL import Image
from tqdm import tqdm
from cnet_dataset_sdxl import HSIControlNetDataset          # 你的 Dataset
from diffusers import (
    StableDiffusionXLControlNetPipeline,
    ControlNetModel,
    UniPCMultistepScheduler,
)
from warp_and_controlnet_model import WarpAndControlNetModel

# ---------------- 配置 ----------------
ROOT_DIR   = "/data/test/"                   # 4 文件夹根目录
CKPT_DIR   = "/data/mv_model_0_with_drop/checkpoint-13500/controlnet/"  # 你的 48 通道 ControlNet 权重
COMPARE_DIR = "./compare_raw"
os.makedirs(COMPARE_DIR, exist_ok=True)

DEVICE  = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- 模型 ----------------
controlnet = WarpAndControlNetModel.from_pretrained(
    CKPT_DIR, torch_dtype=torch.float16, use_safetensors=True,low_cpu_mem_usage=False,device_map=None
)
controlnet = controlnet.to(torch.float16) 
pipe = StableDiffusionXLControlNetPipeline.from_pretrained(
    "hotshotco/SDXL-512",
    controlnet=controlnet,
    torch_dtype=torch.float16,
    use_safetensors=True,
    safety_checker=None,
    num_inference_steps=20,
    guidance_scale=3.0
)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
pipe = pipe.to(DEVICE)
pipe.enable_model_cpu_offload()

# ---------------- 数据集 ----------------
dataset = HSIControlNetDataset(root_dir=ROOT_DIR, image_size=512)

# ---------------- 工具：48→RGB 可视化 ----------------
def vis48(ch48):
    ch48 = ch48.astype(np.float32)
    ch48 = (ch48 - ch48.min()) / (ch48.max() - ch48.min() + 1e-6)
    # 简单取前 3 通道
    rgb = ch48[:3].transpose(1, 2, 0)
    rgb = (rgb * 255).astype(np.uint8)
    return Image.fromarray(rgb)


Warper weights loaded from /data/mv_model_0_with_drop/checkpoint-13500/controlnet/warper.pt


Keyword arguments {'safety_checker': None, 'num_inference_steps': 20, 'guidance_scale': 3.0} are not expected by StableDiffusionXLControlNetPipeline and will be ignored.
Loading pipeline components...: 100%|██████████| 7/7 [00:02<00:00,  2.56it/s]


dataset name: /data/test/
[Dataset] 加载缓存...
folders len 16000
groups  len 4000


In [5]:

# ---------------- 批量推理 + 对比图 ----------------
for idx in tqdm(range(10), desc="Compare"):
    sample = dataset[idx]

    # GT：4-in-1 render_3d.png 拼成的 512×512
    gt_tensor  = sample["pixel_values"]              # (3,512,512) [-1,1]
    gt_img = Image.fromarray(
        ((gt_tensor.permute(1, 2, 0).numpy() + 1) * 127.5).astype(np.uint8)
    )

    # 条件：4-in-1 HSI 48 通道拼成的 512×512
    cond_tensor = sample["conditioning_pixel_values"]  # (48,512,512)
    cond_img = vis48(cond_tensor.numpy())

    # 推理
    # cond_tensor = torch.from_numpy(cond_np).unsqueeze(0).half().to(DEVICE)
    # pred = pipe(
    #     prompt=sample["text"],
    #     # negative_prompt="low quality, blurry",
    #     image=cond_tensor.unsqueeze(0).half().to(DEVICE),
    #     # added_cond_kwargs={"text_embeds": sample["text_embeds"], "time_ids": sample["time_ids"]},
    #     # num_inference_steps=20,
    #     # generator=torch.Generator(DEVICE).manual_seed(42),
    # ).images[0]

    # from torch.cuda.amp import autocast

    # with autocast(dtype=torch.float16):
    pred = pipe(
        prompt=sample["text"],
        image=cond_tensor.unsqueeze(0).to(DEVICE),  # 不用手动 .half()
    ).images[0]

    # 三拼：GT | 条件 | 推理结果
    w, h = 512, 512
    canvas = Image.new("RGB", (w * 3, h))
    canvas.paste(gt_img,    (0, 0))
    canvas.paste(cond_img,  (w, 0))
    canvas.paste(pred,      (w * 2, 0))

    canvas.save(os.path.join(COMPARE_DIR, f"{idx:04d}_gt_cond_pred.png"))

print(f"✅ 全部对比图已保存到 {COMPARE_DIR}")

Compare: 100%|██████████| 10/10 [01:59<00:00, 11.92s/it]

✅ 全部对比图已保存到 ./compare_raw


In [6]:
def show_dtype(model, name=""):
    for n, m in model.named_modules():
        if list(m.children()):        # 跳过空壳容器
            continue
        for p_name, p in m.named_parameters(recurse=False):
            print(f"{name+'/' if name else ''}{n}.{p_name}: {p.dtype}  shape:{tuple(p.shape)}")


show_dtype(pipe.controlnet, "controlnet")

controlnet/conv_in.weight: torch.float16  shape:(320, 4, 3, 3)
controlnet/conv_in.bias: torch.float16  shape:(320,)
controlnet/time_embedding.linear_1.weight: torch.float16  shape:(1280, 320)
controlnet/time_embedding.linear_1.bias: torch.float16  shape:(1280,)
controlnet/time_embedding.linear_2.weight: torch.float16  shape:(1280, 1280)
controlnet/time_embedding.linear_2.bias: torch.float16  shape:(1280,)
controlnet/add_embedding.linear_1.weight: torch.float16  shape:(1280, 2816)
controlnet/add_embedding.linear_1.bias: torch.float16  shape:(1280,)
controlnet/add_embedding.linear_2.weight: torch.float16  shape:(1280, 1280)
controlnet/add_embedding.linear_2.bias: torch.float16  shape:(1280,)
controlnet/controlnet_cond_embedding.conv_in.weight: torch.float16  shape:(16, 48, 3, 3)
controlnet/controlnet_cond_embedding.conv_in.bias: torch.float16  shape:(16,)
controlnet/controlnet_cond_embedding.blocks.0.weight: torch.float16  shape:(16, 16, 3, 3)
controlnet/controlnet_cond_embedding.blocks.